In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# SHORT CALL STRATEGY ON VIXY OPTIONS
#
# Entry:
#   - choose expiration with DTE closest to TARGET_DTE
#   - choose CALL with delta closest to +0.50
#   - sell 1 call
#
# Exit / roll:
#   - hold exact option ID
#   - roll when remaining DTE <= ROLL_DTE
#   - try same-day re-entry if possible
#
# PnL for a short call:
#   position_value_t = - call_mid_t * 100 * contracts
#   daily_pnl = position_value_t - position_value_{t-1}
# ============================================================

FILE = "sample_data/california_housing_train.csv"

TARGET_DTE = 30
MIN_ENTRY_DTE = 12
ROLL_DTE = 7
CALL_TARGET_DELTA = 0.50

CONTRACTS = 1
MULTIPLIER = 100
COMMISSION_PER_CONTRACT_PER_LEG = 0.00

START_DATE = None   # e.g. "2022-08-29"
END_DATE = None     # e.g. "2025-08-29"

# ------------------------------------------------------------
# Load and clean
# ------------------------------------------------------------
df = pd.read_csv(FILE, low_memory=False)

df["date"] = pd.to_datetime(df["date"])
df["exdate"] = pd.to_datetime(df["exdate"])

if START_DATE is not None:
    df = df[df["date"] >= pd.to_datetime(START_DATE)].copy()
if END_DATE is not None:
    df = df[df["date"] <= pd.to_datetime(END_DATE)].copy()

df = df[df["contract_size"] == 100].copy()
df = df[
    df["best_bid"].notna() &
    df["best_offer"].notna() &
    df["delta"].notna()
].copy()

df["mid"] = (df["best_bid"] + df["best_offer"]) / 2.0
df["dte"] = (df["exdate"] - df["date"]).dt.days

df = df[(df["dte"] > 0) & (df["best_offer"] >= df["best_bid"])].copy()
df = df.sort_values(["date", "exdate", "cp_flag", "strike_price"]).reset_index(drop=True)

# Keep calls only for this strategy
calls_df = df[df["cp_flag"] == "C"].copy()

# quote lookup by optionid
quotes_by_option = {
    oid: grp[["date", "mid"]].drop_duplicates("date").set_index("date").sort_index()
    for oid, grp in calls_df.groupby("optionid")
}

calendar = sorted(calls_df["date"].unique())

# ------------------------------------------------------------
# Entry selection
# ------------------------------------------------------------
def select_short_call(day_df, target_dte=30, min_entry_dte=12, call_target=0.50):
    """
    Choose:
      - expiration with DTE >= min_entry_dte and closest to target_dte
      - call with delta closest to +0.50
    """
    if day_df.empty:
        return None

    exp_table = (
        day_df.groupby("exdate", as_index=False)["dte"]
        .first()
        .sort_values("exdate")
    )

    exp_table = exp_table[exp_table["dte"] >= min_entry_dte].copy()
    if exp_table.empty:
        return None

    exp_table["dte_dist"] = (exp_table["dte"] - target_dte).abs()
    chosen_exdate = exp_table.sort_values(["dte_dist", "dte"]).iloc[0]["exdate"]

    sub = day_df[day_df["exdate"] == chosen_exdate].copy()
    if sub.empty:
        return None

    sub["delta_dist"] = (sub["delta"] - call_target).abs()
    chosen = sub.sort_values(["delta_dist", "strike_price"]).iloc[0]

    return {
        "entry_date": chosen["date"],
        "exdate": chosen["exdate"],
        "entry_dte": int(chosen["dte"]),
        "optionid": int(chosen["optionid"]),
        "strike": chosen["strike_price"] / 1000.0,
        "delta": float(chosen["delta"]),
        "mid": float(chosen["mid"]),
    }

# ------------------------------------------------------------
# Backtest
# ------------------------------------------------------------
daily_rows = []
trades = []

in_position = False
current_trade = None
last_position_value = None
trade_id = 0

def open_trade(entry):
    global in_position, current_trade, last_position_value, trade_id
    trade_id += 1
    in_position = True
    current_trade = {
        "trade_id": trade_id,
        **entry
    }
    # Short call value
    last_position_value = -entry["mid"] * MULTIPLIER * CONTRACTS

for current_date in calendar:
    day_df = calls_df[calls_df["date"] == current_date].copy()

    # --------------------------------------------------------
    # Enter if flat
    # --------------------------------------------------------
    if not in_position:
        entry = select_short_call(
            day_df,
            target_dte=TARGET_DTE,
            min_entry_dte=MIN_ENTRY_DTE,
            call_target=CALL_TARGET_DELTA
        )

        if entry is None:
            daily_rows.append({
                "date": current_date,
                "trade_open": False,
                "trade_id": np.nan,
                "position_value": 0.0,
                "daily_pnl": 0.0,
                "call_px": np.nan,
                "dte": np.nan
            })
            continue

        open_trade(entry)

        entry_cost = COMMISSION_PER_CONTRACT_PER_LEG * CONTRACTS
        daily_rows.append({
            "date": current_date,
            "trade_open": True,
            "trade_id": current_trade["trade_id"],
            "optionid": current_trade["optionid"],
            "strike": current_trade["strike"],
            "delta": current_trade["delta"],
            "call_px": current_trade["mid"],
            "position_value": last_position_value,
            "daily_pnl": -entry_cost,
            "dte": (current_trade["exdate"] - current_date).days
        })
        continue

    # --------------------------------------------------------
    # Mark current short call
    # --------------------------------------------------------
    optionid = current_trade["optionid"]
    exdate = current_trade["exdate"]
    remaining_dte = (exdate - current_date).days

    prev_row = daily_rows[-1] if daily_rows else None

    call_px = np.nan
    if optionid in quotes_by_option and current_date in quotes_by_option[optionid].index:
        call_px = float(quotes_by_option[optionid].loc[current_date, "mid"])

    # carry forward if missing
    if np.isnan(call_px) and prev_row is not None:
        call_px = prev_row.get("call_px", np.nan)

    if np.isnan(call_px):
        position_value = last_position_value
        daily_pnl = 0.0
    else:
        position_value = -call_px * MULTIPLIER * CONTRACTS
        daily_pnl = position_value - last_position_value
        last_position_value = position_value

    daily_rows.append({
        "date": current_date,
        "trade_open": True,
        "trade_id": current_trade["trade_id"],
        "optionid": optionid,
        "strike": current_trade["strike"],
        "delta": current_trade["delta"],
        "call_px": call_px,
        "position_value": position_value,
        "daily_pnl": daily_pnl,
        "dte": remaining_dte
    })

    # --------------------------------------------------------
    # Roll
    # --------------------------------------------------------
    if remaining_dte <= ROLL_DTE:
        exit_cost = COMMISSION_PER_CONTRACT_PER_LEG * CONTRACTS
        daily_rows[-1]["daily_pnl"] -= exit_cost

        trade_rows = [r for r in daily_rows if r.get("trade_id") == current_trade["trade_id"]]
        trade_pnl = sum(r["daily_pnl"] for r in trade_rows)

        trades.append({
            "trade_id": current_trade["trade_id"],
            "entry_date": current_trade["entry_date"],
            "exit_date": current_date,
            "exdate": current_trade["exdate"],
            "entry_dte": current_trade["entry_dte"],
            "exit_dte": remaining_dte,
            "optionid": current_trade["optionid"],
            "strike": current_trade["strike"],
            "entry_delta": current_trade["delta"],
            "trade_pnl": trade_pnl
        })

        in_position = False
        current_trade = None
        last_position_value = None

        # same-day re-entry
        same_day_entry = select_short_call(
            day_df,
            target_dte=TARGET_DTE,
            min_entry_dte=MIN_ENTRY_DTE,
            call_target=CALL_TARGET_DELTA
        )

        if same_day_entry is not None and same_day_entry["entry_dte"] > ROLL_DTE:
            open_trade(same_day_entry)
            entry_cost = COMMISSION_PER_CONTRACT_PER_LEG * CONTRACTS

            daily_rows.append({
                "date": current_date,
                "trade_open": True,
                "trade_id": current_trade["trade_id"],
                "optionid": current_trade["optionid"],
                "strike": current_trade["strike"],
                "delta": current_trade["delta"],
                "call_px": current_trade["mid"],
                "position_value": last_position_value,
                "daily_pnl": -entry_cost,
                "dte": (current_trade["exdate"] - current_date).days
            })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
daily = pd.DataFrame(daily_rows)
trades_df = pd.DataFrame(trades)

if not daily.empty:
    daily = daily.sort_values(["date", "trade_id"], na_position="last").reset_index(drop=True)
    daily["daily_pnl"] = pd.to_numeric(daily["daily_pnl"], errors="coerce").fillna(0.0)
    daily["cum_pnl"] = daily["daily_pnl"].cumsum()

print("\n===== PERFORMANCE SUMMARY =====")
if not daily.empty:
    print(f"Total PnL ($): {daily['cum_pnl'].iloc[-1]:.2f}")
    print(f"Completed trades: {len(trades_df)}")
    print(f"Days with open trade: {int(daily['trade_open'].fillna(False).sum())}")
    print(f"Days flat: {int((~daily['trade_open'].fillna(False)).sum())}")

if not trades_df.empty:
    print("\n===== TRADE SUMMARY =====")
    print(trades_df.to_string(index=False))

# Cumulative PnL
plt.figure(figsize=(10, 5))
plt.plot(daily["date"].to_numpy(), daily["cum_pnl"].to_numpy())
plt.title("Short Call Strategy on VIXY")
plt.xlabel("Date")
plt.ylabel("Cumulative PnL ($)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Call price of current held position
held = daily[daily["trade_open"] == True].copy()
if not held.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(held["date"].to_numpy(), held["call_px"].to_numpy())
    plt.title("Held Call Mid Price Over Time")
    plt.xlabel("Date")
    plt.ylabel("Call Mid Price")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'b10xheebr3v3h1vn.csv'